#### SBERT
- BERT 모델 : 문장 이해용 Encoder
    - 문장 쌍 비교
- SBERT 모델 : 문장 의미 임베딩
    - 벡터의 비교용

In [ ]:
# !pip install sentence-transformers

In [2]:
import torch
from sentence_transformers import SentenceTransformer, util

In [4]:
# 모델을 로드 -> 두개의 문장을 비교(코사인 유사도)
# 다목적 한국 SBERT
model_name = 'jhgan/ko-sroberta-multitask'
# 문장 유사도 특화
model_name2 = 'BM-K/KoSimCSE-roberta-multitask'

sbert = SentenceTransformer(model_name)
sbert2 = SentenceTransformer(model_name2)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [5]:
# 최대 토론의 길이를 설정
sbert.max_seq_length =256
sbert2.max_seq_length = 256

In [6]:
doc1 = "이 카메라는 색감이 자연스럽고 배터리도 오래간다"
doc2 = "배터리 성능이 좋고 사진 품질이 뛰어나다"

In [9]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산
# sbert 인 경우
with torch.inference_mode():
    emb1 = sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim, 4))

유사도 :  0.6948


In [10]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산
# sbert 인 경우
with torch.inference_mode():
    emb1 = sbert2.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert2.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim2 = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim2, 4))

유사도 :  0.734


In [11]:
emb1.shape

torch.Size([768])

In [ ]:
sentences = [
    '삼성전자 주가가 올랐다',
    '코스피가 상승 마감했다',
    '비가 많이 와서 항공편이 취소됐다'
]

with torch.inference_mode():
    embs = sbert2.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)

sim_metrix = util.cos_sim(embs, embs)

In [13]:
print(sim_metrix)

tensor([[ 1.0000,  0.3585, -0.0052],
        [ 0.3585,  1.0000,  0.1199],
        [-0.0052,  0.1199,  1.0000]])


In [14]:
new_sentence = '증시가 강세였다'
# 임베딩
new_emb = sbert2.encode(new_sentence, convert_to_numpy=True, normalize_embeddings=True)
# 유사도가 높은 상위의 n개 확인
top_n = 2
hits = torch.topk(
    util.cos_sim(new_emb, embs).squeeze(0), k = top_n
)
hits

torch.return_types.topk(
values=tensor([0.6161, 0.6014]),
indices=tensor([0, 1]))

In [15]:
for score, idx in zip(hits.values.tolist(), hits.indices.tolist() ):
    print(f"{sentences[idx]} | score {round(score, 3)}")

삼성전자 주가가 올랐다 | score 0.616
코스피가 상승 마감했다 | score 0.601


### 연습
- ratings_train.txt 파일 로드
- 결측치 제거
- documents 컬럼의 문자 정규화(특수문자 제거, 2칸 이상의 공백 제거, 좌우 공백 제거)
- 중복 documents 제거, 글자의 수가 1개 이하인 행은 제거
- DataFrame에서 sample(n = 10000, random_state = 42)로 임의의 데이터를 추출하여 저장(head() -> 상위 데이터 tail() -> 하위 데이터 | sample() -> 무작위 데이터)
- train, test 8:2 로 데이터 분할
- sbert 모델은 'BM-K/KoSimCSE-roberta-multitask' 을 이용
- Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
    - 입력받은 document 와 label을 document는 SBERT 모델을 이용하여 인코딩
    - label 데이터를 tensor형태로 변환
    - '__len__' 함수는 라벨의 길이를 되돌려준다
    - '__getitem__' 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
- Dataset을 train,test를 이용하여 Dataset 생성
- DataLoader를 이용하여 배치의 사이즈는 128 shuffle 은 True로 구성한다.

In [43]:
import re
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [44]:
def normalize_token_text(text : str) -> str:
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [53]:
# 데이터 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.dropna(subset='document',inplace=True)
df['document'] = df['document'].map(normalize_token_text) # 정규화
df.drop_duplicates(subset=['document'] , inplace=True) # 중복데이터 제거
df = df.loc[df['document'].str.len() > 1] # document길이가 1이하면 제거
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 144637 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        144637 non-null  int64 
 1   document  144637 non-null  object
 2   label     144637 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.4+ MB


In [46]:
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [ ]:
df2 = df.sample(n = 10000, random_state=42)
df2.reset_index(drop= True, inplace=True)
train_df, test_df = train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [60]:
model_name2

'BM-K/KoSimCSE-roberta-multitask'

In [61]:
MODEL_NAME = 'BM-K/KoSimCSE-roberta-multitask'
sbert3 = SentenceTransformer(MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [56]:
from torch.utils.data import Dataset, DataLoader

In [63]:
# - Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
#     - '__init__(self, document, labels)'
#     - 입력받은 document 와 label을 document는 SBERT 모델을 이용하여 인코딩
#     - label 데이터를 tensor형태로 변환
#     - '__len__' 함수는 라벨의 길이를 되돌려준다
#     - '__getitem__' 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
#     - 인코딩은 __init__ 함수 안에서 진행한다
class SBERTDataset(Dataset):
    def __init__(self, df, model_name):
        self.documents = df['document'].tolist()
        self.labels = torch.tensor(df['label'].tolist())
        self.sbert = SentenceTransformer(model_name)
        self.sbert.max_seq_length = 256
        with torch.inference_mode():
            self.embeddings = sbert3.encode(
                self.documents, convert_to_tensor=True, normalize_embeddings=True
            )
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


In [64]:
# - Dataset을 train,test를 이용하여 Dataset 생성
train_dataset = SBERTDataset(train_df, MODEL_NAME)
test_dataset = SBERTDataset(test_df, MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.
No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [65]:
# - DataLoader를 이용하여 배치의 사이즈는 128 shuffle 은 True로 구성한다.
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)
# test DataLoader
test_dataloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)
# DataLoader 확인
for batch in train_dataloader:
    inputs, labels = batch
    print("입력 데이터 크기 : ", inputs.shape)
    print("라벨 데이터 크기 : ", labels.shape)
    break
# - Dataset을 train,test를 이용하여 Dataset 생성
# train 데이터셋의 크기
print("train 데이터셋의 크기 : ", len(train_dataset))
# test 데이터셋의 크기
print("test 데이터셋의 크기 : ", len(test_dataset))


입력 데이터 크기 :  torch.Size([128, 768])
라벨 데이터 크기 :  torch.Size([128])
train 데이터셋의 크기 :  8000
test 데이터셋의 크기 :  2000
